# SentinelSleep — Pre-generate Therapeutic Audio Cache on Colab GPU

**Purpose:** Run `scripts/pregenerate_cache.py` on a Colab GPU (T4/L4/A100) to build  
the `data/audio_cache/` directory that is too heavy for the M2 8 GB local machine.  
Download the resulting zip and unpack it locally.

**Runtime:** Set to **GPU** before running (Runtime → Change runtime type → T4 GPU).  
**Time:** ~10–20 min total (model downloads + generation + mixing).

---

## Cell 1 — Clone the repo and pin to a known commit

In [ ]:
# If the repo is private, authenticate first:
#   !git config --global credential.helper store
#   !echo 'https://<YOUR_GH_PAT>:x-oauth-basic@github.com' > ~/.git-credentials

!git clone https://github.com/reddy-nithin/SentinalSleep.git
%cd SentinalSleep

# Pin to the exact commit so manifest.git_commit is meaningful.
# Replace with 'main' if you want the latest.
COMMIT = "main"
!git checkout {COMMIT}
!git log --oneline -3

## Cell 2 — Install uv and sync dependencies

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# Sync all deps into a local .venv
!uv sync
print("uv sync complete")

## Cell 3 — (Optional) Hugging Face authentication

Only needed if you hit rate-limit errors downloading `cvssp/audioldm2`.  
Store your token in **Colab Secrets** (key icon in the left sidebar) as `HF_TOKEN` — do NOT paste it directly in the notebook.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        !uv run huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
        print("HF login successful")
    else:
        print("HF_TOKEN secret not set — proceeding without auth (may hit rate limits)")
except Exception as e:
    print(f"Could not load HF_TOKEN: {e} — proceeding without auth")

## Cell 4 — Run the cache builder

This runs MusicGen (300M params) then AudioLDM2 (1.1B params), then mixes 10 clips.  
Each model is loaded, used, then unloaded before the next loads (memory budget enforcement).  
Expected output: 3 music + 3 soundscape + 10 mixed WAV files + `manifest.json`.

In [ ]:
!uv run python scripts/pregenerate_cache.py
print("\nCache build finished")

## Cell 5 — Verify the cache before downloading

In [ ]:
!uv run python scripts/verify_cache.py --no-sha256
# --no-sha256 is fast enough for a quick sanity check here;
# the full SHA-256 check runs locally after download.

## Cell 6 — Zip and download

In [ ]:
import shutil
shutil.make_archive("audio_cache", "zip", "data", "audio_cache")
print("Zipped → audio_cache.zip")

# Show size
import os
size_mb = os.path.getsize("audio_cache.zip") / 1_048_576
print(f"Size: {size_mb:.1f} MB")

In [ ]:
from google.colab import files
files.download("audio_cache.zip")

---
## After download — run these commands locally

```bash
# 1. Unpack into your local repo
cd /path/to/SentinalSleep
unzip -o ~/Downloads/audio_cache.zip -d data/

# 2. Full integrity check (includes SHA-256)
uv run python scripts/verify_cache.py

# 3. Confirm all tests still pass
uv run pytest tests/ -q
```

If `verify_cache.py` exits 0, Phase 3 is complete and you can start Phase 4 (Orchestration).